# exp-041 — training-seed variance of the headline fine-tune (Colab)

Retrains exp-026's pool-only QLoRA fine-tune (Mistral-7B-Instruct-v0.3, 2,000-question pool, 250 steps) under
training seeds **42, 43 and 44** on this one GPU, and evaluates each exactly as exp-026 was evaluated (fp16 + adapter,
`--no-few-shot`, pinned 600). exp-015's prompting command is re-run on the same GPU as the comparator. Only `--seed`
moves; no pipeline code changes. Design: `experiments/exp-041/README.md`, `run_config.json`, ADR 0047.

**Before you run**
1. Runtime → **A100** (≥ 16 GiB and bf16 required; L4 also passes; T4/V100 are refused).
2. Colab secret `HF_TOKEN` (Mistral licence accepted), or `HF_TOKEN` in the environment.
3. `MyDrive/thesis-backup.tar.gz` exists. The notebook clones the public repo; the first launch pins the commit
   (`MyDrive/thesis-exp-041/launch_sha.txt`), and every training and evaluation snapshot must carry it.

**Order:** setup → 20-question canary of the prompting command → for seeds 43, 44, 42: train (+ gate), 20-question
canary with the adapter, full evaluation (+ gate) → prompting re-run → spread + paired compare + Holm.
Estimate (not measured): **~1.7–2.8 h on an A100**.

**If the runtime dies:** run all cells again. Finished steps on Drive are skipped; an interrupted decode resumes;
an interrupted training restarts from step 0 with the same seed (ADR 0044 item 5).

In [ ]:
# ---- settings ----
REPO_URL = "https://github.com/AhmadiJahid/Thesis---QAv2.git"
BRANCH = "main"
COMMIT = ""                                  # "" = pin the branch head at first launch; or a full 40-hex SHA
ARCHIVE_ON_DRIVE = "/content/drive/MyDrive/thesis-backup.tar.gz"
ARCHIVE_SHA256 = "33c6072f0e4e46ab94f40f804463c512f439823b80c3e0e9dec0ee1023717d94"   # as in exp-037
DRIVE_OUT = "/content/drive/MyDrive/thesis-exp-041"
WORK = "/content/exp041"
VENV = "/content/venv313_exp041"
PY = f"{VENV}/bin/python"
SEEDS = [43, 44, 42]                         # the two new seeds first; 42 = exp-026's seed re-trained on this GPU

In [ ]:
# ---- Drive (skipped when already mounted, e.g. a runner executing this notebook with nbconvert) ----
import os, json, hashlib, subprocess, shutil, time, datetime, glob, re, threading
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(f"{DRIVE_OUT}/logs", exist_ok=True)
os.makedirs(f"{DRIVE_OUT}/runs/exp-041", exist_ok=True)
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(ARCHIVE_ON_DRIVE), f"archive not found: {ARCHIVE_ON_DRIVE}"

def sh(cmd, log=None, env=None, cwd=None, check=True):
    # Run a shell command, stream its output here and (optionally) append it to a Drive log.
    print(f"$ {cmd}", flush=True)
    full_env = {**os.environ, **(env or {})}
    with subprocess.Popen(cmd, shell=True, cwd=cwd, env=full_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash") as p:
        fh = open(log, "a") if log else None
        if fh: fh.write(f"\n### {datetime.datetime.now().isoformat()} $ {cmd}\n")
        for line in p.stdout:
            if "Processed" not in line:
                print(line, end="", flush=True)
            if fh: fh.write(line); fh.flush()
        rc = p.wait()
        if fh: fh.write(f"### rc={rc}\n"); fh.close()
    if check and rc != 0:
        raise RuntimeError(f"command failed (rc={rc}): {cmd}")
    return rc

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
print("Drive OK; outputs ->", DRIVE_OUT)

In [ ]:
# ---- clone (public repo); the FIRST launch pins its commit on Drive, every later launch checks it out ----
REPO = f"{WORK}/Thesis---QAv2"
if not os.path.isdir(f"{REPO}/.git"):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch -q origin {BRANCH}")
PIN_FILE = f"{DRIVE_OUT}/launch_sha.txt"
pinned = open(PIN_FILE).read().strip() if os.path.exists(PIN_FILE) else None
if COMMIT and pinned and COMMIT != pinned:
    raise SystemExit(f"COMMIT={COMMIT} differs from the pinned {pinned} ({PIN_FILE}); finished steps on Drive "
                     "came from the pinned commit - use it, or start a fresh DRIVE_OUT")
SHA = COMMIT or pinned or subprocess.check_output(["git", "-C", REPO, "rev-parse", f"origin/{BRANCH}"], text=True).strip()
assert re.fullmatch(r"[0-9a-f]{40}", SHA), SHA
sh(f"git -C {REPO} checkout -q --detach {SHA}")
assert subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip() == SHA
if not pinned:
    open(PIN_FILE, "w").write(SHA + "\n")
assert not subprocess.check_output(["git", "-C", REPO, "status", "--porcelain", "--untracked-files=no"],
                                   text=True).strip(), "clone is not clean"
os.chdir(REPO)
print("pinned commit:", SHA)

In [ ]:
# ---- GPU gate (total VRAM; another job may share the GPU - timings in the README assume it does not) ----
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,compute_cap,driver_version",
                             "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
name, mem_mib, used_mib, cc, driver = [x.strip() for x in q.split(",")]
print(f"GPU {name}: {int(mem_mib)/1024:.1f} GiB total, {int(used_mib)/1024:.1f} GiB in use, sm_{cc}, driver {driver}")
assert int(mem_mib) / 1024 >= 16, "need >= 16 GiB VRAM: Runtime -> Change runtime type -> A100"
cc_t = tuple(int(x) for x in cc.split("."))
assert cc_t >= (8, 0), f"{name} (sm_{cc}) has no bf16: use an A100 or L4 runtime"
if "A100" not in name:
    print("NOTE: not an A100 - the README's time estimates are for an A100")
GPU_INFO = {"name": name, "memory_total_mib": int(mem_mib), "driver": driver, "compute_cap": cc}
open(f"{DRIVE_OUT}/logs/gpu.txt", "a").write(f"{datetime.datetime.now().isoformat()} {q}\n")

In [ ]:
# ---- Python 3.13.9 venv with the repo's pins (the exp-026 environment, docs/compute.md "Colab") ----
if not os.path.exists(f"{VENV}/.exp041_ok"):
    sh("pip -q install uv")
    sh(f"uv venv --python 3.13.9 {VENV}")
    sh(f"uv pip install --python {PY} torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128")
    sh(f"uv pip install --python {PY} -r {REPO}/requirements.txt")
    open(f"{VENV}/.exp041_ok", "w").write("ok")
ENV_JSON = f"{DRIVE_OUT}/logs/environment.json"
sh(f'''{PY} - <<'EOF' > {ENV_JSON}
import json, sys, torch, transformers, peft, trl, bitsandbytes, accelerate, datasets, huggingface_hub, sentence_transformers
print(json.dumps({{"python": sys.version.split()[0], "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
  "cuda_available": torch.cuda.is_available(), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
  "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__,
  "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__, "datasets": datasets.__version__,
  "huggingface_hub": huggingface_hub.__version__, "sentence_transformers": sentence_transformers.__version__}}, indent=2))
EOF''')
env_rec = json.load(open(ENV_JSON)); print(json.dumps(env_rec, indent=1))
want = {"python": "3.13.9", "torch": "2.10.0", "transformers": "5.5.4", "peft": "0.20.0", "trl": "1.10.0",
        "bitsandbytes": "0.49.2", "accelerate": "1.12.0", "datasets": "4.8.4", "huggingface_hub": "1.11.0",
        "sentence_transformers": "5.2.3"}
off = {k: (env_rec[k], v) for k, v in want.items() if not str(env_rec[k]).startswith(v)}
assert env_rec["cuda_available"], "CUDA not available inside the venv"
assert not off, f"version drift from requirements.txt: {off}"
os.environ["PY"] = PY
print("environment matches requirements.txt")

In [ ]:
# ---- Hugging Face token (Mistral-7B-Instruct-v0.3 is gated): environment first, then Colab secrets ----
if not os.environ.get("HF_TOKEN"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
sh(f'{PY} -c "from huggingface_hub import whoami; print(\'HF user:\', whoami()[\'name\'])"')

In [ ]:
# ---- data: private copy of the archive (atomic, so a concurrent notebook never reads a half copy),
#      then scripts/colab_setup_data.py extracts only what exp-041 reads (safe extraction, sha-checked) ----
LOCAL_TAR = f"/content/thesis-backup.exp-041.tar.gz"
if not os.path.exists(LOCAL_TAR):
    t = time.time(); shutil.copy(ARCHIVE_ON_DRIVE, LOCAL_TAR + ".part"); os.replace(LOCAL_TAR + ".part", LOCAL_TAR)
    print(f"copied archive in {time.time()-t:.0f}s")
sh(f"{PY} scripts/colab_setup_data.py --archive {LOCAL_TAR} --sha256 {ARCHIVE_SHA256} --parts musique "
   "--runs pool_sweep/pools/size2000_imbalanced_trial0_poolseed42/pool.jsonl exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl exp-032/exp-015/eval exp-032/exp-015/reparse exp-032/exp-026/eval exp-032/exp-026/reparse", log=f"{DRIVE_OUT}/logs/setup_data.log")
# runs/exp-041 IS the Drive folder: every finished row (results.partial.jsonl) is on Drive at once.
link = "runs/exp-041"
if not os.path.islink(link):
    assert not os.path.exists(link), f"{link} exists and is not the Drive link"
    os.symlink(f"{DRIVE_OUT}/runs/exp-041", link)
print(link, "->", os.readlink(link))

In [ ]:
# ---- step runner: experiments/exp-041/run.sh <step>, log streamed to Drive, VRAM sampled ----
def run_step(*step):
    tag = "_".join(step)
    log_path = f"{DRIVE_OUT}/logs/{tag}.log"
    peak = {"mib": 0}; stop = threading.Event()
    def sampler():
        while not stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True).stdout.strip()
            if out: peak["mib"] = max(peak["mib"], int(out.splitlines()[0]))
            stop.wait(10)
    th = threading.Thread(target=sampler, daemon=True); th.start()
    started = time.time()
    rc = sh("bash experiments/exp-041/run.sh " + " ".join(step), log=log_path, env={"PY": PY}, check=False)
    stop.set(); th.join()
    rec = {"step": tag, "rc": rc, "wall_seconds": round(time.time() - started),
           "peak_vram_mib_sampled_whole_gpu": peak["mib"], "gpu": GPU_INFO, "git_sha": SHA,
           "finished": datetime.datetime.now().isoformat()}
    with open("runs/exp-041/colab_steps.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(rec)
    assert rc == 0, f"{tag} failed (rc={rc}; rc=3 is a failed output-health gate); see {log_path}. " \
                    "Do not re-run with changed settings: a changed run is a new experiment."

## 1. Canary: 20 questions of exp-015's command (environment, model, generation)

In [ ]:
run_step("canary_prompted")

## 2. Per seed: train (gate) → 20-question canary with the adapter → full evaluation (gate) → evaluator

In [ ]:
for s in SEEDS:
    run_step("train", str(s))
    tg = json.load(open(f"runs/exp-041/seed{s}/train_gate.json"))
    print(f"seed {s} training gate:", tg["ok"])
    run_step("eval", str(s))
    em = json.load(open(f"runs/exp-041/seed{s}/eval/eval_metrics.json"))
    print(f"seed {s} report card:", {k: round(em[k], 4) for k in ("break_exact_match_rate", "sari_macro", "ged_macro",
          "chain_validity_macro", "hop_count_exact_match_rate", "under_decomposition_rate", "over_decomposition_rate")})

## 3. exp-015's prompting command on this GPU (the same-machine comparator)

In [ ]:
run_step("prompted")

## 4. Spread + paired compare + Holm (CPU)

In [ ]:
run_step("compare")
shutil.copy("experiments/exp-041/compare_metrics.json", f"{DRIVE_OUT}/compare_metrics.json")
m = json.load(open("experiments/exp-041/compare_metrics.json"))
for k, v in m["hardware_control"].items():
    print(f"hardware control {k} vs {v['vs']}: {v['identical_decompositions']}/{v['items']} identical")
ag = m["agreement_primary_vs_reference"]
print("cells holding across all seeds:", ag["cells_holding_across_all_seeds"], "/", ag["counts_out_of"])
print(json.dumps(ag["counts"], indent=1))

In [ ]:
# ---- what to copy back to the laptop ----
print(f"Everything is in {DRIVE_OUT}/ on Google Drive. Copy back (repo = C:/Users/suuser/thesis/Thesis---QAv2):")
print(f"  {DRIVE_OUT}/runs/exp-041/          -> runs/exp-041/   (gitignored; adapters, results, per-item files)")
print(f"  {DRIVE_OUT}/compare_metrics.json   -> experiments/exp-041/compare_metrics.json   (committed)")
print(f"  {DRIVE_OUT}/logs/                  -> runs/exp-041/colab_logs/")
print(f"  {DRIVE_OUT}/launch_sha.txt         -> the commit for the log row ({SHA})")
print("Then copy each training config.json/metrics.json and decomposer config.json into experiments/exp-041/,")
print("write notes.md, and complete the exp-041 row in experiments/log.md (README 'After the run').")
for root, _, files in os.walk("runs/exp-041", followlinks=True):
    if "checkpoints" in root or "prompts_log" in root:
        continue
    for f in files:
        if f.endswith((".json", ".jsonl", ".md")):
            print("   ", os.path.join(root, f))